<h2 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Introduction and Problem Statement
</font>
</h2>


<p dir=ltr style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir">
In this section of the competition, our goal is to classify automotive consumable parts based on their images. These supplies can include items such as motor oil, air filters, brake pads, and other similar parts and items.
You must design and train an artificial intelligence model capable of receiving an image of each part and placing it in the correct category. For this purpose, a collection of sample images from each category will be provided to you so that you can train your model based on them.
</font>
</p>

<h2 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Dataset Introduction
</font>
</h2>

<p dir=ltr style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
In the initial folder of this question, there is a folder named <code>Data</code> which contains 2 folders, <code>train</code> and <code>test</code>.
Each folder contains the images for the training and testing sets.
</font>
</p>

<p dir=ltr style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
All training images are located in the training folder. In this folder, there is another file named <code>train.csv</code> in which the label of each image is specified in <code>One_Hot</code> format.
</font>
</p>


<p dir=ltr style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>
All test images are also located in the testing folder. Additionally, a file named <code>test.csv</code> is placed in this folder. This file contains the names of the test set images, but their labels are unspecified. You must predict the label for each image in the order of the image names in this file.
</font>
</p>

<h4 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Training Dataset Introduction (train)
</font>
</h4>

<div dir="ltr">
<p dir=ltr style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>

The <code>train</code> set contains `969` images with dimensions of `640x640`.

The <code>train.csv</code> file is a file in which the name of each image and its label are specified in `one_hot` format. This file is structured as follows:

<center>
<div dir=ltr style="direction: ltr;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    

| Column Name | Description |
|:---: |:---: |
| `filename` | Image name |
| `air_filter` | Whether the image category is an air filter. |
| `brake_pad` | Whether the image category is a brake pad. |
| `coolant` | Whether the image category is coolant. |
| `oil_filter` | Whether the image category is an oil filter. |
| `oil_motor` | Whether the image category is motor oil. |
| `windshield_washer` | Whether the image category is windshield washer fluid. |


</font>
</div>
</center>
</div>

<h4 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
Test Dataset Introduction (test)
</font>
</h4>

<div dir="ltr">
<p dir=ltr style="direction: ltr; text-align: justify; line-height:200%; font-family:vazir; font-size:medium">
<font face="vazir" size=3>

The <code>test</code> set contains `356` images with dimensions of `640x640`.

In the <code>test.csv</code> file, the names of the test set images can be found. Ultimately, you must make your predictions based on this file. This file is structured as follows:

<center>
<div dir=ltr style="direction: ltr;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    

| Column Name | Description |
|:---: |:---: |
| `filename` | Image name |


</font>
</div>
</center>
</div>

## Library Imports

Import all required libraries: `os`, `random`, `numpy`, `pandas`, `PIL` for I/O and data handling; `torch`, `torch.nn`, `torch.utils.data` for model building and data loading; `torchvision.transforms` and `torchvision.models` for image augmentation and pretrained-backbone access; `sklearn` for stratified splitting and evaluation metrics.

In [ ]:
import os
from typing import List
import zipfile
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import f1_score, classification_report, confusion_matrix

## Global Hyperparameters & Device Configuration

Define all training hyperparameters in one place for easy tuning:
- `SEED` — random seed for reproducibility
- `BATCH_SIZE` — number of samples per gradient-update step
- `LR` / `WEIGHT_DECAY` — AdamW learning rate and L2 regularisation strength
- `EPOCHS` — total training passes over the dataset
- `IMG_SIZE` — spatial resolution each image is resized to before entering the backbone
- `BACKBONE` — selects the CNN architecture (ResNet50 is active; others are commented out for quick experimentation)
- `device` — auto-detects GPU; falls back to CPU if CUDA is unavailable

In [15]:
SEED = 42
DATA_ROOT = "./"
BATCH_SIZE = 32
NUM_WORKERS = 2
LR = 3e-4
WEIGHT_DECAY = 1e-4
EPOCHS = 50

IMG_SIZE = 224
# BACKBONE = "vit_b_16"

# IMG_SIZE = 320
# BACKBONE = "resnet18"
# BACKBONE = "resnet34"
BACKBONE = "resnet50"
# BACKBONE = "convnext_tiny"
# BACKBONE = "efficientnet_b0"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = torch.cuda.is_available()

## Load CSV Manifests

Read the train and test CSV files that map image filenames to one-hot class labels. `train.csv` contains labelled examples used for training and validation; `test.csv` lists test-set filenames whose labels must be predicted.

In [16]:
train_csv_path = os.path.join(DATA_ROOT, "train", "train.csv")
test_csv_path  = os.path.join(DATA_ROOT, "test", "test.csv")

train_df = pd.read_csv(train_csv_path)
test_df  = pd.read_csv(test_csv_path)

## Inspect Training Data

Quick sanity-check: display the first five rows of the training DataFrame to verify column names, filename format, and label encoding.

In [17]:
train_df.head()

,filename,air_filter,brake_pad,coolant,oil_filter,oil_motor,windshield_washer
0,1088_JPG.rf.5ea3233c11949cb830d74d79264aae49.jpg,0,1,0,0,0,0
1,784_jpeg.rf.98eb9c0935510870e583070eb783f2c5.jpg,1,0,0,0,0,0
2,451_jpg.rf.034ef8f9cd4dc93142253e0ece06ed08.jpg,0,0,0,0,1,0
3,1190_jpg.rf.4483dd98fa193f81bebfc5f5bd5883aa.jpg,0,0,1,0,0,0
4,400_jpg.rf.c82fc29b487e59b5da2856ebd8962e52.jpg,0,0,1,0,0,0


## Inspect Test Data

Display the first five rows of the test DataFrame to confirm the expected structure (filename column only — no ground-truth labels).

In [18]:
test_df.head()

,filename
0,603_jpg.rf.622bda703e651e344eca8148cc0576c0.jpg
1,61_jpg.rf.f9c76a4acc90b6025bd17f92717cd55a.jpg
2,342_jpg.rf.7abe7b58cbf130a6d2db1c390b782370.jpg
3,539_jpg.rf.296e885aba52a70ebce9f2491a4d13eb.jpg
4,219_jpeg.rf.5ef5b8a4f57aa608413f0c7bccf56c64.jpg


## Label Parsing & Type Coercion

Identify class columns (every column that is not `filename`), cast them to `int` to ensure numeric label values, and strip whitespace from filename strings to prevent path-lookup errors at dataset load time.

In [19]:
class_cols = [c for c in train_df.columns if c.lower() != "filename"]

for c in class_cols:
    train_df[c] = train_df[c].astype(int)

train_df["filename"] = train_df["filename"].astype(str).str.strip()
test_df["filename"]  = test_df["filename"].astype(str).str.strip()

## Augmentation Pipelines, Dataset Class & Data Loaders

**Augmentation transforms:**
- `train_tfms` applies random horizontal flip, ±10° rotation, and colour jitter (brightness, contrast, saturation, hue) for regularisation, then resizes, converts to tensor, and applies ImageNet normalisation (`mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`).
- `eval_tfms` skips stochastic augmentations — only resize, tensor conversion, and normalisation — to ensure deterministic validation and test predictions.

**`CarPartDataset` (PyTorch `Dataset`):**
- `__init__`: stores paths and transform pipeline; resets the DataFrame index for clean integer indexing.
- `__len__`: returns the number of samples.
- `__getitem__`: loads an RGB image via Pillow, applies the transform pipeline, and converts the one-hot label row to a scalar class index via `argmax` (returns `-1` for test samples that have no labels).

**Stratified split:** `StratifiedShuffleSplit` partitions the training data 85 / 15 so every class is proportionally represented in both train and validation splits.

**Class-weight computation:** Inverse-frequency weights are computed from training-set class counts and later passed to `CrossEntropyLoss` to up-weight rare car-part classes.

**DataLoaders** wrap each split's dataset with shuffling for train, pinned memory for fast GPU transfer, and multiple worker processes for parallel data loading.

In [20]:
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

class CarPartDataset(Dataset):
    def __init__(self, df: pd.DataFrame, root_dir: str, img_dirname: str,
                 class_cols: List[str], tfms, drop_missing=True):
        self.root_dir = root_dir
        self.img_dirname = img_dirname
        self.class_cols = class_cols
        self.tfms = tfms

        df = df.reset_index(drop=True).copy()
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, self.img_dirname, row["filename"])
        img = Image.open(img_path).convert("RGB")
        img = self.tfms(img)

        if all(c in self.df.columns for c in self.class_cols):
            y = int(np.argmax(row[self.class_cols].values))
        else:
            y = -1

        return img, y


y_idx = train_df[class_cols].values.argmax(axis=1)
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
tr_idx, va_idx = next(sss.split(train_df, y_idx))
tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
va_df = train_df.iloc[va_idx].reset_index(drop=True)

class_counts = np.bincount(tr_df[class_cols].values.argmax(axis=1), minlength=len(class_cols))
class_weights = (1.0 / np.maximum(class_counts, 1))
class_weights = class_weights / class_weights.sum() * len(class_cols)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

print("Class counts:", class_counts.tolist())
print("Class weights:", np.round(class_weights, 4).tolist())

train_ds = CarPartDataset(tr_df, DATA_ROOT, "train", class_cols, train_tfms, drop_missing=True)
val_ds   = CarPartDataset(va_df, DATA_ROOT, "train", class_cols, eval_tfms,  drop_missing=True)
test_ds  = CarPartDataset(test_df, DATA_ROOT, "test",  class_cols, eval_tfms,  drop_missing=True)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=False
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=False
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=False
)


Class counts: [131, 105, 110, 139, 228, 110]
Class weights: [0.9762, 1.2179, 1.1625, 0.92, 0.5609, 1.1625]


## Model Factory — `make_model`

A single factory function that instantiates a backbone from `torchvision.models` **without** pretrained weights (`weights=None`, i.e. trained from scratch) and replaces its final classification head with a new `nn.Linear` layer whose output dimension equals the number of car-part classes.

Supported backbones: ResNet-18/34/50, EfficientNet-B0, ConvNeXt-Tiny, ViT-B/16. The ResNet variants expose `model.fc`; EfficientNet and ConvNeXt expose `model.classifier[-1]`; ViT exposes `model.heads.head`.

In [21]:
def make_model(num_classes: int, backbone: str = "resnet50") -> nn.Module:
    b = backbone.lower()
    if b == "resnet18":
        m = models.resnet18(weights=None)
        in_feats = m.fc.in_features
        m.fc = nn.Linear(in_feats, num_classes)

    elif b == "resnet34":
        m = models.resnet34(weights=None)
        in_feats = m.fc.in_features
        m.fc = nn.Linear(in_feats, num_classes)

    elif b == "resnet50":
        m = models.resnet50(weights=None)
        in_feats = m.fc.in_features
        m.fc = nn.Linear(in_feats, num_classes)

    elif b == "efficientnet_b0":
        m = models.efficientnet_b0(weights=None)
        in_feats = m.classifier[-1].in_features
        m.classifier[-1] = nn.Linear(in_feats, num_classes)

    elif b == "convnext_tiny":
        m = models.convnext_tiny(weights=None)

        in_feats = m.classifier[-1].in_features
        m.classifier[-1] = nn.Linear(in_feats, num_classes)

    elif b == "vit_b_16":
        m = models.vit_b_16(weights=None)

        in_feats = m.heads.head.in_features
        m.heads.head = nn.Linear(in_feats, num_classes)

    else:
        print("error model 404")

    return m

model = make_model(num_classes=len(class_cols), backbone=BACKBONE).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, EPOCHS))

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0

    for step, batch in enumerate(loader):
        try:
            imgs, ys = batch
        except Exception as e:
            print(f"[ERR] Failed to fetch batch at step {step}: {repr(e)}")
            raise

        imgs, ys = imgs.to(device), ys.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, ys)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * imgs.size(0)
        preds = logits.argmax(1)
        correct += (preds == ys).sum().item()
        total += imgs.size(0)

    return loss_sum / max(total, 1), correct / max(total, 1)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_ys = [], []

    for imgs, ys in loader:
        imgs = imgs.to(device)
        logits = model(imgs)
        preds = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_ys.extend(ys.numpy().tolist())

    macro_f1 = f1_score(all_ys, all_preds, average="macro")
    weighted_f1 = f1_score(all_ys, all_preds, average="weighted")
    acc = (np.array(all_preds) == np.array(all_ys)).mean()

    return macro_f1, weighted_f1, acc, np.array(all_ys), np.array(all_preds)

BEST_PATH = "best_model.pt"
best_f1, best_epoch = -1.0, -1

for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    macro_f1, weighted_f1, va_acc, ys_true, ys_pred = evaluate(model, val_loader, device)
    scheduler.step()

    print(f"Epoch {epoch:02d} | loss {tr_loss:.4f} | tr_acc {tr_acc:.4f} "
          f"| val_acc {va_acc:.4f} | val_macroF1 {macro_f1:.4f}")

    if macro_f1 > best_f1:
        best_f1, best_epoch = macro_f1, epoch
        torch.save({
            "state_dict": model.state_dict(),
            "backbone": BACKBONE,
            "img_size": IMG_SIZE,
            "class_cols": class_cols
        }, BEST_PATH)

print(f"Best macro-F1: {best_f1:.4f} at epoch {best_epoch}")


Epoch 01 | loss 1.7837 | tr_acc 0.2831 | val_acc 0.1301 | val_macroF1 0.0534
Epoch 02 | loss 1.4961 | tr_acc 0.4374 | val_acc 0.4726 | val_macroF1 0.3192
Epoch 03 | loss 1.1482 | tr_acc 0.5796 | val_acc 0.4589 | val_macroF1 0.4422
Epoch 04 | loss 1.0526 | tr_acc 0.5942 | val_acc 0.5959 | val_macroF1 0.5878
Epoch 05 | loss 0.9603 | tr_acc 0.6343 | val_acc 0.5548 | val_macroF1 0.5545
Epoch 06 | loss 0.8800 | tr_acc 0.6768 | val_acc 0.3904 | val_macroF1 0.3137
Epoch 07 | loss 0.8101 | tr_acc 0.7060 | val_acc 0.5890 | val_macroF1 0.5524
Epoch 08 | loss 0.7419 | tr_acc 0.7145 | val_acc 0.6507 | val_macroF1 0.6415
Epoch 09 | loss 0.6681 | tr_acc 0.7327 | val_acc 0.7123 | val_macroF1 0.7069
Epoch 10 | loss 0.6126 | tr_acc 0.7594 | val_acc 0.6918 | val_macroF1 0.6809
Epoch 11 | loss 0.6023 | tr_acc 0.7837 | val_acc 0.6918 | val_macroF1 0.6714
Epoch 12 | loss 0.5767 | tr_acc 0.8080 | val_acc 0.6849 | val_macroF1 0.6740
Epoch 13 | loss 0.4412 | tr_acc 0.8275 | val_acc 0.8082 | val_macroF1 0.7940

## Load Best Checkpoint & Evaluate on Validation Set

Load the best checkpoint saved during training, reconstruct the model architecture using the backbone name stored in the checkpoint, and restore learned weights via `load_state_dict`. Then run the `evaluate` function on the held-out validation loader and print overall accuracy, macro-averaged F1, weighted F1, a per-class classification report, and the confusion matrix.

In [25]:
ckpt = torch.load(BEST_PATH, map_location=device)
best_model = make_model(num_classes=len(class_cols), backbone=ckpt.get("backbone", BACKBONE)).to(device)
best_model.load_state_dict(ckpt["state_dict"])

macro_f1, weighted_f1, va_acc, ys_true, ys_pred = evaluate(best_model, val_loader, device)

print("Validation Accuracy:", round(va_acc, 4))
print("Macro F1:", round(macro_f1, 4))
print("Weighted F1:", round(weighted_f1, 4))

print("\nClassification Report (per-class):")
print(classification_report(ys_true, ys_pred, digits=4, target_names=class_cols))

cm = confusion_matrix(ys_true, ys_pred)
print("\nConfusion Matrix:")
print(cm)


Validation Accuracy: 0.8699
Macro F1: 0.8568
Weighted F1: 0.8679

Classification Report (per-class):
                   precision    recall  f1-score   support

       air_filter     0.8571    0.7826    0.8182        23
        brake_pad     0.8889    0.8421    0.8649        19
          coolant     0.8235    0.7368    0.7778        19
       oil_filter     0.8519    0.9200    0.8846        25
        oil_motor     0.8889    0.9756    0.9302        41
windshield_washer     0.8889    0.8421    0.8649        19

         accuracy                         0.8699       146
        macro avg     0.8665    0.8499    0.8568       146
     weighted avg     0.8690    0.8699    0.8679       146


Confusion Matrix:
[[18  1  0  4  0  0]
 [ 3 16  0  0  0  0]
 [ 0  0 14  0  3  2]
 [ 0  0  1 23  1  0]
 [ 0  1  0  0 40  0]
 [ 0  0  2  0  1 16]]


<h2 dir=ltr align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
 Prediction for Test Data and Output
</font>
</h2>

<p dir=ltr style="direction: ltr;text-align: left;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    Save your model's predictions on the test data (<code>test</code>) as a <code>dataframe</code> in a variable named <code>submission</code>. This <code>dataframe</code> must have a column named <code>filename</code> where its <code>i</code>-th row is your prediction for the <code>i</code>-th row of the test dataset.
    Ultimately, you must predict which category your image belongs to, and your <code>submission.csv</code> file should look like the following:
</font>
</p>

<center>
<div dir=ltr style="direction: ltr;line-height:200%;font-family:vazir;font-size:medium">
<font face="vazir" size=3>
    
|windshield_washer|oil_motor|oil_filter|coolant|brake_pad|air_filter|filename|
|------|------|---|------|---|------|---|
|0|0|0|0|1|0|123.jpg|
|0|1|0|0|0|0|124.jpg|
    
</font>

</div>
</center>

## Inference on Test Set & Submission CSV Generation

`predict` runs the model in evaluation mode with gradient computation disabled (`@torch.no_grad()`), collects softmax probability vectors for every test batch, and concatenates them into a single array.

The predicted class index (`argmax` over probabilities) is converted back to a one-hot row via `np.eye`, combined with the test filenames, reordered to match the expected submission column order, and written to `submission.csv`.

In [26]:
@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    probs = []

    for x, _ in loader:
        x = x.to(device)
        probs.append(torch.softmax(model(x), dim=1).cpu().numpy())

    return np.concatenate(probs)


probs = predict(best_model, test_loader, device)
preds = probs.argmax(axis=1)

oh = np.eye(len(class_cols))[preds]

submission = pd.DataFrame(oh, columns=class_cols)
submission["filename"] = test_df["filename"]

desired_order = [
    "filename",
    "air_filter",
    "brake_pad",
    "coolant",
    "oil_filter",
    "oil_motor",
    "windshield_washer"
]

submission = submission[[c for c in desired_order if c in submission.columns]]
submission.to_csv("submission.csv", index=False)

## Package Submission Artefacts

Write the final `submission.csv`, then compress it together with `notebook.ipynb` into `result.zip` using DEFLATE compression — the format required by the competition judge.

In [ ]:
if not os.path.exists(os.path.join(os.getcwd(), 'notebook.ipynb')):
    %notebook -e notebook.ipynb

def compress(file_names):
    print("File Paths:")
    print(file_names)
    compression = zipfile.ZIP_DEFLATED
    with zipfile.ZipFile("result.zip", mode="w") as zf:
        for file_name in file_names:
            zf.write('./' + file_name, file_name, compress_type=compression)

submission.to_csv('submission.csv', index=False)
file_names = ['notebook.ipynb', 'submission.csv']
compress(file_names)

File Paths:
['notebook.ipynb', 'submission.csv']
